In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

class MelSegmentDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.X = np.load(x_path)
        self.y = np.load(y_path)
        # Resize to 227x227 as required by AlexNet, using bilinear interpolation [cite: 206, 207]
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((227, 227), antialias=True)
        ])

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        # Transpose from (C, H, W) to (H, W, C) for the transform
        img = self.X[idx].transpose(1, 2, 0)
        return self.transform(img).float(), torch.tensor(self.y[idx], dtype=torch.long)

# Load Data
dataset = MelSegmentDataset("processed_data_emodb/X_segments.npy", "processed_data_emodb/y_labels.npy")
loader = DataLoader(dataset, batch_size=30, shuffle=True) # Batch size of 30 [cite: 410]

# Initialize pre-trained AlexNet [cite: 149]
model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Modify the final layer for 7 EMO-DB classes instead of 1000 [cite: 245]
num_ftrs = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_ftrs, 7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Training setup (SGD, lr=0.001, momentum=0.9) [cite: 410]
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Fine-tuning Loop
model.train()
for epoch in range(10): # Set to 300 for full replication [cite: 411]
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(loader)}")

# Save fine-tuned weights
torch.save(model.state_dict(), "fine_tuned_alexnet.pth")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /Users/leyanzhi/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100.0%


Epoch 1, Loss: 2.1238865286764437
Epoch 2, Loss: 1.9072237824001452
Epoch 3, Loss: 1.9082918210621298
Epoch 4, Loss: 1.9079084448570753
Epoch 5, Loss: 1.9069917523947946
Epoch 6, Loss: 1.9075208745733665
Epoch 7, Loss: 1.9074070183900151
Epoch 8, Loss: 1.9073656388442881
Epoch 9, Loss: 1.9072101307611395
Epoch 10, Loss: 1.9067188101093264
